<a href="https://colab.research.google.com/github/sanmeshh/pytorch_learning/blob/12.RNN_Pytorch/RNN_Pytorch_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df=pd.read_csv('/content/100_Unique_QA_Dataset.csv')


In [ ]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [ ]:
#tokenzing

def tokenize(text):
  text=text.lower()
  text=text.replace('?',"")
  text=text.replace("'","")
  return text.split()



In [ ]:
#forming vocab
vocab={'<UNK>':0}
def build_vocab(row):
  q=tokenize(row['question'])
  a=tokenize(row['answer'])
  merged_tokens=q+a
  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)



df.apply(build_vocab,axis=1)




,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [ ]:
len(vocab)

324

In [ ]:
# convert words to numbers indices

def text_to_indices(text,vocab):
  indexed_text=[]

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text



In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_q=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_a=text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_q),torch.tensor(numerical_a)

In [ ]:
dataset=QADataset(df,vocab)

In [ ]:
dataloader=DataLoader(dataset,batch_size=1)
#no need of padding since batch_size=1


In [ ]:
import torch.nn as nn

In [ ]:
'''here we are not using Sequential container
because self.rnn gives two outputs
the hidden states(vectors) for a particular input in the rnn
the output for that particular input from the rnn'''

class SimpleRNN(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()

    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)#the shape[0] is the batch size

    self.fc=nn.Linear(64,vocab_size)

  def forward(self,question):

    embedded=self.embedding(question)
    hidden,final=self.rnn(embedded)
    output=self.fc(final.squeeze(0))#1,1,324->1,324
    return output




In [ ]:
lr=0.001
epochs=20


In [ ]:
model=SimpleRNN(len(vocab))

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=lr)

In [ ]:
for epoch in range(epochs):
  total_loss=0

  for question,answer in dataloader:

    optimizer.zero_grad()

    #forward pass
    output=model(question)
    #loss
    loss=criterion(output,answer[0])


    #gradient
    loss.backward()

    #update
    optimizer.step()

    total_loss=total_loss+loss.item()

  print(f"Epoch:{epoch+1}, Loss:{total_loss:4f}")



Epoch:1, Loss:528.521505
Epoch:2, Loss:461.603216
Epoch:3, Loss:383.923000
Epoch:4, Loss:319.306704
Epoch:5, Loss:266.605596
Epoch:6, Loss:218.307274
Epoch:7, Loss:174.454641
Epoch:8, Loss:136.277135
Epoch:9, Loss:104.820995
Epoch:10, Loss:80.276533
Epoch:11, Loss:61.878416
Epoch:12, Loss:48.352511
Epoch:13, Loss:38.427204
Epoch:14, Loss:31.065076
Epoch:15, Loss:25.497174
Epoch:16, Loss:21.198743
Epoch:17, Loss:17.825344
Epoch:18, Loss:15.141855
Epoch:19, Loss:12.982748
Epoch:20, Loss:11.229067


In [ ]:
def predict(model,question,threshold=0.1):

  numerical_q=text_to_indices(question,vocab)

  #'tensorify'
  tensor_q=torch.tensor(numerical_q).unsqueeze(0)

  #send to model

  output=model(tensor_q)

  #converts logits to probs
  probs=torch.nn.functional.softmax(output,dim=1)

  #find index of max prob
  value,index=torch.max(probs,dim=1)

  if value<threshold:
    print("IDK")
  else:
    print((list(vocab.keys())[index]).title())


In [ ]:
predict(model,"Who directed the movie 'Titanic'")

Jamescameron
